In [93]:
import pandas as pd
import os
from nilearn import datasets,image, surface, plotting
import numpy as np
Dir_github = '/work/svs-lab/xuanyang/Project/NSF_valence/github'
repo = 'ABC_valence'
pipeline = 'compareROI'
Dir_output = os.path.join(Dir_github,'derivatives','fMRI',pipeline,'ROI')
if not os.path.exists(Dir_output):
    os.makedirs(Dir_output)
# /work/svs-lab/xuanyang/Project/NSF_valence/github/derivatives/fMRI/compareROI/
# /scripts/fMRI/master/
df_jhu = pd.read_excel(os.path.join(Dir_github,repo,'scripts','fMRI','master','ROI_jhu.xlsx'),header=None,
                       names=['idx','regionName','regionLongName','Matter','Labels'])
# df_jhu.dropna(subset='Labels')

df_jhu['LR'] = ''
df_jhu.loc[df_jhu['regionName'].str.endswith('_R'),'LR'] = 'R'
df_jhu.loc[df_jhu['regionName'].str.endswith('_L'),'LR'] = 'L'
df_jhu.loc[df_jhu['regionName'].str.contains('_L_'),'LR'] = 'L'
df_jhu.loc[df_jhu['regionName'].str.contains('_R_'),'LR'] = 'R'
df_jhu['Labels'].unique()
df_jhu

,idx,regionName,regionLongName,Matter,Labels,LR
0,1,SFG_L,superior frontal gyrus (posterior segment) left,1,NaN,L
1,2,SFG_R,superior frontal gyrus (posterior segment) right,1,NaN,R
2,3,SFG_PFC_L,superior frontal gyrus (prefrontal cortex) left,1,mPFC,L
3,4,SFG_PFC_R,superior frontal gyrus (prefrontal cortex) right,1,mPFC,R
4,5,SFG_pole_L,superior frontal gyrus (frontal pole) left,1,mPFC,L
...,...,...,...,...,...,...
184,185,PSTG_R,posterior superior temporal gyrus right,1,STG,R
185,186,PSMG_L,posterior middle temporal gyrus left,1,NaN,L
186,187,PSMG_R,posterior middle temporal gyrus right,1,NaN,R
187,188,PSIG_L,posterior inferior temporal gyrus left,1,NaN,L


In [94]:
df_glasser = pd.read_excel(os.path.join(Dir_github,repo,'scripts','fMRI','master','ROI_glasser.xlsx'))
df_glasser['Labels'].unique()
# df_glasser.dropna(subset='Labels')

array([nan, 'PCC', 'STG', 'hippocampus', 'parahippocampus', 'precuneus',
       'pgACC', 'mPFC', 'sgACC', 'OFC'], dtype=object)

In [70]:
Dir_mask = '/work/xy6/templates'


img_jhu = image.load_img(os.path.join(Dir_mask,'jhu.nii'))
data_jhu = img_jhu.get_fdata()

img_glasser = image.load_img(os.path.join(Dir_mask,'MNI_Glasser_HCP_v1.0.nii.gz'))
data_glasser = img_glasser.get_fdata()


In [95]:
List_ROIs = ['PCC', 'STG', 'hippocampus', 'parahippocampus', 'precuneus',
       'pgACC', 'mPFC', 'sgACC', 'OFC']
for roi in List_ROIs:
    for atlas, df,data,img_tmp in zip(['jhu','glasser'],[df_jhu,df_glasser],[data_jhu,data_glasser],[img_jhu,img_glasser,]):
        df_tmp = df.loc[df['Labels']==roi].copy()
        data_output = data.copy()
        data_output[np.isin(data_output,df_tmp['idx'],invert=True)] = 0
        data_output[data_output>1000] = data_output[data_output>1000]-1000
        # binary
        new_img = image.new_img_like(img_tmp, data_output)
        output_path = os.path.join(Dir_output, f"ROI_{roi}_{atlas}.nii.gz")
        new_img.to_filename(output_path)
